# `semantic.v_beat_rate_by_cut` — view

Thin view over Gold. No logic beyond shaping.

A view is its own definition, so there is no load step and no etl task to pair with this one.

In [ ]:
-- THE FIVE CUTS. One view, one extra column saying which cut a row belongs to, because
-- five near-identical views would be five things to maintain and explain.
--
-- Four of them are a GROUP BY over a key the fact or dim_ticker already holds. The manager
-- cut is the one place a trust can be counted more than once.
CREATE OR REPLACE VIEW `index-vs-trust-pipeline`.semantic.v_beat_rate_by_cut
COMMENT 'Beat rate by management group, manager, region, sole/multi manager, and AIC sector'
AS
WITH trusts AS (
  SELECT f.ticker_key, f.management_group_key, f.mandate_key,
         f.horizon_years, f.beat_index, f.total_return, f.volatility,
         f.risk_adjusted_return,
         d.manager_structure, d.aic_sector
  FROM `index-vs-trust-pipeline`.semantic.v_horizon_performance f
  JOIN `index-vs-trust-pipeline`.gold.dim_ticker d
    ON d.ticker_key = f.ticker_key AND d.entity_type = 'Trust'
),
ticker_manager AS (
  -- The fact carries manager_key, so the trust-to-manager link is read off the fact rather
  -- than out of the bridge. Same pairs either way; this is the one the star describes.
  -- The horizon rows are already collapsed, so DISTINCT is what re-attaches the managers.
  SELECT DISTINCT ticker_key, manager_key
  FROM `index-vs-trust-pipeline`.gold.fact_monthly_performance
),
cut AS (
  -- The house comes from its own dimension rather than a string on dim_ticker. Joined
  -- here and not in `trusts`, so this branch cannot take the others down with it.
  SELECT 'management group' AS cut_by, g.management_group AS cut_value,
         t.horizon_years, t.beat_index, t.total_return, t.volatility, t.risk_adjusted_return
  FROM trusts t
  JOIN `index-vs-trust-pipeline`.gold.dim_management_group g
    ON g.management_group_key = t.management_group_key
  UNION ALL
  -- IMPACT BASIS. A trust with six managers is counted six times, once for each of them.
  -- That is right for "what share of managers had a trust that beat the index" and wrong
  -- for anything that has to reconcile to the 440 horizon rows -- weight by the fact's
  -- allocation_factor for that.
  SELECT 'manager', m.manager_name,
         t.horizon_years, t.beat_index, t.total_return, t.volatility, t.risk_adjusted_return
  FROM trusts t
  JOIN ticker_manager tm                                  ON tm.ticker_key = t.ticker_key
  JOIN `index-vs-trust-pipeline`.gold.dim_manager m       ON m.manager_key = tm.manager_key
  -- The two special members are placeholders for "no person", not managers to rank.
  WHERE m.manager_name NOT IN ('NotApplicable', 'NoInfo')
  UNION ALL
  -- WHERE the money goes. The cut aic_sector could never give on its own: eight buckets
  -- instead of thirty-two, because the sector string mixes geography with asset class.
  SELECT 'region', n.region,
         t.horizon_years, t.beat_index, t.total_return, t.volatility, t.risk_adjusted_return
  FROM trusts t
  JOIN `index-vs-trust-pipeline`.gold.dim_mandate n ON n.mandate_key = t.mandate_key
  UNION ALL
  SELECT 'manager structure', t.manager_structure,
         t.horizon_years, t.beat_index, t.total_return, t.volatility, t.risk_adjusted_return
  FROM trusts t
  UNION ALL
  SELECT 'aic sector', t.aic_sector,
         t.horizon_years, t.beat_index, t.total_return, t.volatility, t.risk_adjusted_return
  FROM trusts t
)
SELECT cut_by,
       cut_value,
       horizon_years,
       COUNT(*)                                                   AS trusts,
       SUM(CASE WHEN beat_index THEN 1 ELSE 0 END)                AS beat_count,
       ROUND(100.0 * SUM(CASE WHEN beat_index THEN 1 ELSE 0 END)
             / COUNT(*), 1)                                       AS beat_rate_pct,
       ROUND(100 * PERCENTILE_APPROX(total_return, 0.5), 1)       AS median_return_pct,
       ROUND(100 * PERCENTILE_APPROX(volatility, 0.5), 1)         AS median_volatility_pct,
       ROUND(PERCENTILE_APPROX(risk_adjusted_return, 0.5), 2)     AS median_risk_adjusted
FROM cut
GROUP BY cut_by, cut_value, horizon_years;

## Verification

Expected: **five** values in `cut_by`, and the `manager` cut carrying more trust-rows than
the others because a multi-manager trust is counted once per manager.

On `management group`, `region`, `manager structure` and `aic sector` every horizon must
total the same number of trusts as `v_beat_rate` — **75 / 83 / 89 / 89 / 89**. If one does
not, its key has not been filled and the join is dropping rows.

In [ ]:
SELECT cut_by,
       COUNT(*)              AS rows_in_cut,
       COUNT(DISTINCT cut_value) AS distinct_values,
       SUM(trusts)           AS trust_rows_counted
FROM `index-vs-trust-pipeline`.semantic.v_beat_rate_by_cut
GROUP BY cut_by
ORDER BY cut_by;